<h1 align='center'>VQA-E Master Guide</h1>
<h3 align='center'>Generative Visual Question Answering — LSTM Decoder with Explanation</h3>
<p align='center'>
  <b>Production Models: E (ConvNeXt) · F (BUTD)</b><br>
  Architecture reference · Data preparation · Training scripts · Evaluation
</p>

> **How to use this notebook:**  
> - **Guide mode** (recommended): read sections 2–7 for architecture understanding, then run training via shell scripts.  
> - **Interactive mode**: run code cells in sequence for step-by-step exploration.  
> - **For actual training → use the shell scripts** (`train_model_e.sh`, `train_model_f.sh`).


---
# 1. Environment Setup


## 1.1 Install Dependencies

```bash
# Core
pip install torch torchvision nltk tqdm matplotlib Pillow

# NLP metrics
pip install rouge-score

# Experiment tracking (optional)
pip install wandb

# Tier D5: hallucination filter (optional)
pip install spacy
python -m spacy download en_core_web_sm

# Tier 9: ConceptNet GNN (optional — MLP fallback works without it)
pip install torch_geometric
```


In [13]:
# Run this cell to install everything needed
import subprocess, sys
pkgs = ['torch','torchvision','nltk','tqdm','matplotlib','Pillow','rouge-score']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print('✅ Dependencies installed')


✅ Dependencies installed


## 1.2 GPU & Environment Check


In [14]:
import torch, os, sys

# ── GPU ──────────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    vram = p.total_memory / 1024**3
    print(f"GPU     : {p.name}")
    print(f"VRAM    : {vram:.1f} GB")
    print(f"SM      : {p.major}.{p.minor}  ({p.multi_processor_count} SMs)")
    bf16 = torch.cuda.is_bf16_supported()
    print(f"BF16    : {bf16}  {'← native Tensor Core (Ampere+)' if bf16 else ''}")
    print(f"SDPA    : {hasattr(torch.nn.functional, 'scaled_dot_product_attention')}")
    print(f"Compile : {hasattr(torch, 'compile')}")
    print()

    # Hardware-tuned recommendations
    if p.major >= 12:
        arch = "Blackwell"
    elif p.major >= 9:
        arch = "Ada Lovelace"
    elif p.major >= 8:
        arch = "Ampere"
    else:
        arch = f"SM {p.major}.{p.minor}"

    if vram >= 20:
        batch_e, batch_f = 256, 256
    elif vram >= 14:
        batch_e, batch_f = 128, 192
    elif vram >= 10:
        batch_e, batch_f = 64, 128
    else:
        batch_e, batch_f = 32, 64

    print(f"Architecture     : {arch}")
    print(f"Recommended (E)  : --batch_size {batch_e}  (ConvNeXt in forward pass)")
    print(f"Recommended (F)  : --batch_size {batch_f}  (pre-extracted BUTD, no CNN)")
    print(f"Num workers      : {min(os.cpu_count(), 12)}  (I/O-bound optimum)")
    print()
    print("AMP mode :", "BFloat16 (no GradScaler)" if bf16 else "Float16 + GradScaler")
    print()
    print("Run benchmark for exact tuning:")
    print("  python src/scripts/benchmark_5070ti.py --quick")
else:
    print("No CUDA GPU found.")

# ── Python / PyTorch ──────────────────────────────────────────────────────────
print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")

# ── Verify imports ────────────────────────────────────────────────────────────
sys.path.insert(0, 'src')
try:
    from models.vqa_models import VQAModelE, VQAModelF
    from training.losses import SequenceFocalLoss
    from training.curriculum import CurriculumSampler
    print("\nCore imports    : OK")
except ImportError as e:
    print(f"\nImport warning  : {e}")


GPU     : NVIDIA GeForce RTX 5070 Ti
VRAM    : 15.5 GB
SM      : 12.0  (70 SMs)
BF16    : True  ← native Tensor Core (Ampere+)
SDPA    : True
Compile : True

Architecture     : Blackwell
Recommended (E)  : --batch_size 128  (ConvNeXt in forward pass)
Recommended (F)  : --batch_size 192  (pre-extracted BUTD, no CNN)
Num workers      : 12  (I/O-bound optimum)

AMP mode : BFloat16 (no GradScaler)

Run benchmark for exact tuning:
  python src/scripts/benchmark_5070ti.py --quick
Python  : 3.11.15
PyTorch : 2.10.0+cu128

Core imports    : OK


## 1.3 Project Structure

```
new_vqa/
├── src/
│   ├── models/
│   │   ├── encoder_cnn.py          # SimpleCNN, ResNet, ConvNeXt, BUTDFeatureEncoder
│   │   ├── encoder_question.py     # BiLSTM + Char-CNN + Highway
│   │   ├── decoder_lstm.py         # LSTMDecoder (Models A/B)
│   │   ├── decoder_attention.py    # MHCA + LSTMDecoderWithAttention (C/D/E/F)
│   │   ├── pointer_generator.py    # Pointer-Generator Network (Tier 5)
│   │   ├── vqa_models.py           # VQAModelA/B/C/D/E/F + GatedFusion + MUTANFusion
│   │   └── concept_gnn.py          # ConceptNet GNN (Tier 9, optional)
│   ├── training/
│   │   ├── losses.py               # SequenceFocalLoss (Tier D3)
│   │   ├── curriculum.py           # Question-type CurriculumSampler (Tier D4)
│   │   ├── css_augment.py          # CSS Counterfactual Augmentation (Tier 6)
│   │   └── scst.py                 # SCST Reinforcement Learning (Tier 8)
│   ├── scripts/
│   │   ├── 1_build_vocab.py        # Build vocab_questions + vocab_answers JSONs
│   │   ├── extract_butd_features.py # Extract Faster R-CNN features (Tier 3B)
│   │   ├── filter_hallucinations.py # Filter bad VQA-E explanations (Tier D5)
│   │   └── generate_synthetic_qa.py # Template synthetic QA (Tier D5)
│   ├── train.py                    # Main entry — all flags, all phases
│   ├── evaluate.py                 # 7-metric evaluation
│   ├── inference.py                # Greedy + beam search decoding
│   ├── compare.py                  # Multi-model comparison
│   ├── plot_curves.py              # Training curves
│   └── visualize.py                # Attention heatmaps
├── data/
│   ├── raw/train2014/              # COCO train images (~82K)
│   ├── raw/val2014/                # COCO val images (~40K)
│   ├── raw/vqa_data_json/          # VQA v2.0 JSON files
│   ├── vqa_e/                      # VQA-E annotation JSONs
│   ├── processed/                  # vocab_questions.json, vocab_answers.json
│   └── butd_features/              # Pre-extracted BUTD .pt files (Model F)
└── checkpoints/                    # Saved model weights + training history
```


---
# 2. Architecture Deep Dive


## 2.1 The 6 Models

### Production Models (current expansion — recommended)

| Model | Image Encoder | Fusion | Decoder | Status |
|-------|--------------|--------|---------|--------|
| **E** | ConvNeXt-Base (ImageNet-1K pretrained) | **MUTAN** | LSTM + **MHCA** + PGN | **Primary target** |
| **F** | BUTD (pre-extracted Faster R-CNN 2048-d) | **MUTAN** | LSTM + **MHCA** + img_mask | **Best ceiling** |

Model E trains end-to-end (ConvNeXt fine-tuned in Phase 2).  
Model F uses cached object features — highest accuracy ceiling but requires `extract_butd_features.py` first.

### Baseline Models (legacy — do not retrain)

| Model | Image Encoder | Fusion | Decoder | Baseline BLEU-4 |
|-------|--------------|--------|---------|-----------------|
| A | SimpleCNN (scratch) | GatedFusion | LSTM (no attention) | ~5% |
| B | ResNet101 (pretrained) | GatedFusion | LSTM (no attention) | ~7% |
| C | SimpleCNN + 49 regions | GatedFusion | LSTM + Bahdanau | ~9% |
| D | ResNet101 + 49 regions | GatedFusion | LSTM + Bahdanau | ~11.6% |

> Models A–D are included in `vqa_models.py` for ablation comparison only.  
> All new training should use Model **E** or **F**.

### Data Flow (Model E/F)

```
Image (3,224,224) → ConvNeXt-Base → (B, 49, 1024) spatial features
Question tokens   → BiLSTM+Highway+CharCNN → q_feature (B, H), q_hidden (B, L, H)
                    → MUTANFusion(q_feature, img_mean) → h_0 for LSTM decoder
                    → LSTMDecoderWithAttention (teacher forcing)
                       ├─ img_mhca: Q=h_t, K/V=img_features, with coverage + img_mask
                       └─ q_mhca:  Q=h_t, K/V=q_hidden
                    → PointerGeneratorHead → mixed P_vocab + P_copy
                    → Output: "yellow because bananas are yellow" token-by-token
```


## 2.2 Multi-Head Cross-Attention (MHCA) — The Legal Attention

**Why not self-attention?** Self-attention inside `decode_step` would run
O(T · (L_v² + L_q²)) per forward pass — quadratic in sequence length, repeated
at every decode step. Also violates the no-transformer constraint.

**MHCA is cross-attention only**:
- Q = h_t ∈ ℝᴴ — LSTM hidden state (single token, not a sequence)
- K, V = memory ∈ ℝˢˣᴴ — image regions or question tokens
- Complexity: O(T · S) — linear in memory length

```
Q = W_Q(h_t)                  (B, 1, H)
K = W_K(memory)               (B, S, H)
V = W_V(memory)               (B, S, H)

Reshape to heads: (B, nh, ·, d_head) where d_head = H / nh

scores = Q·Kᵀ / √d_head       (B, nh, 1, S)
[img]  + cov_scale · coverage  ← optional coverage bias
α      = softmax(scores)       (B, nh, 1, S)
context = α · V                (B, H)   after merging heads
α_mean  = mean(α, heads)       (B, S)   → P_copy for PGN
```


## 2.3 MUTAN Tucker Fusion (Model E/F)

Replaces GatedFusion with a Tucker-decomposed bilinear product:

```
q_proj = tanh(W_q · q)   ∈ ℝᵗ_q
v_proj = tanh(W_v · v)   ∈ ℝᵗ_v
inter  = q_proj ⊗ T_c    ∈ ℝᵗ_v × d_out    (T_c ∈ ℝᵗ_q × ᵗ_v × d_out  learnable core)
output = v_proj · inter  ∈ ℝᵈ_out           (rank-constrained bilinear)
       → BatchNorm
```

**Why MUTAN?** GatedFusion is a gated weighted average — it can only mix, not
multiply. MUTAN captures multiplicative cross-modal interactions (e.g. 'red ball'
= color × object) via Tucker decomposition without the parameter explosion of
a full bilinear product (H² parameters → t_q × t_v parameters, t_q=t_v=360).


## 2.4 Pointer-Generator Network (PGN)

Allows the decoder to **copy tokens from the question** when the answer
contains the same words (e.g. Q: 'What color is the **red** car?' → A: '**red**').

```
p_gen  = σ(W_x·embed_t + W_h·h_t + W_ctx·img_context)  ∈ (0,1)

P_vocab = softmax(logit_t)             ← generate from vocabulary
P_copy  = scatter(q_α, q_token_ids)   ← copy from question (q_α from q_mhca)

P_final = p_gen · P_vocab + (1-p_gen) · P_copy
loss    = NLLLoss(log(P_final), target)   ← NLL not CE (log already applied)
```


## 2.5 Sequence Focal Loss (Tier D3)

**The problem with class-weighted CE**: `CrossEntropyLoss(weight=w)` applies
the same scalar w[c] at EVERY position. The word 'because' (very common in
VQA-E) would have w=0.01, teaching the model to skip the structural hinge.

**Focal Loss is position-aware**:
```
ce_t         = -log P(y_t | context)          per token, per position
p_t          = exp(-ce_t)                      model confidence (0→hard, 1→easy)
focal_loss_t = (1 - p_t)^γ · ce_t             down-weight easy tokens naturally

loss = Σ_{t: y_t ≠ pad} focal_loss_t / |valid tokens|
```
Common tokens ('because', 'the') predicted with high p_t → suppressed.
Rare answer words where model struggles (low p_t) → full gradient.


## 2.6 All Training Techniques at a Glance

| Flag | Tier | What it does |
|------|------|-------------|
| `--layer_norm` | 1A | LayerNorm inside LSTM cell gates |
| `--dropconnect` | 1B | DropConnect on hidden-to-hidden weights (AWD-LSTM) |
| `--dcan` | 2→MHCA | No-op now; MHCA always active |
| `--coverage` | - | Coverage mechanism: penalizes re-attending to regions |
| `--pgn` | 5 | Pointer-Generator: copy tokens from question |
| `--css` | 6 | CSS: visual + linguistic counterfactual contrastive loss |
| `--q_highway` | 7B | Highway connections between BiLSTM layers |
| `--char_cnn` | 7C | Character-level CNN prepended to word embeddings |
| `--scst` | 8 | SCST RL: REINFORCE with BLEU-4 reward (Phase 4) |
| `--focal` | D3 | SequenceFocalLoss replaces CrossEntropyLoss |
| `--curriculum` | D4 | Question-type curriculum: binary→color→count→what→where→why |
| `--mix_vqa` | D2 | 70% VQA v2.0 + 30% VQA-E to prevent length bias |
| `--scheduled_sampling` | - | Reduce exposure bias (epsilon-greedy teacher forcing) |
| `--glove` | - | GloVe 300d pretrained word embeddings |


In [15]:
# Instantiate all 6 models and print parameter counts
import sys; sys.path.insert(0, 'src')
import torch
from models.vqa_models import VQAModelA, VQAModelB, VQAModelC, VQAModelD, VQAModelE, VQAModelF

VOCAB_Q, VOCAB_A = 8000, 4000

configs = [
    ('A', VQAModelA, {}),
    ('B', VQAModelB, {}),
    ('C', VQAModelC, {'use_coverage': True}),
    ('D', VQAModelD, {'use_coverage': True}),
    ('E', VQAModelE, {'use_coverage': True, 'use_mutan': True, 'use_dcan': True}),
    ('F', VQAModelF, {'use_coverage': True, 'use_mutan': True, 'feat_dim': 1029}),
]

print(f"{'Model':<8} {'Params':>12}  Description")
print('-' * 60)
descs = {
    'A': 'SimpleCNN + GatedFusion + LSTMDecoder',
    'B': 'ResNet101 + GatedFusion + LSTMDecoder',
    'C': 'SimpleCNNSpatial + GatedFusion + LSTM+MHCA',
    'D': 'ResNet101Spatial + GatedFusion + LSTM+MHCA',
    'E': 'ConvNeXt-Base + MUTANFusion + LSTM+MHCA',
    'F': 'BUTD(Faster R-CNN) + MUTANFusion + LSTM+MHCA',
}
for name, cls, kw in configs:
    try:
        m = cls(vocab_size=VOCAB_Q, answer_vocab_size=VOCAB_A, **kw)
        p = sum(x.numel() for x in m.parameters() if x.requires_grad)
        print(f'  Model {name}  {p:>12,}  {descs[name]}')
    except Exception as e:
        print(f'  Model {name}  [error: {e}]')


Model          Params  Description
------------------------------------------------------------
  Model A    43,388,928  SimpleCNN + GatedFusion + LSTMDecoder
  Model B    38,162,944  ResNet101 + GatedFusion + LSTMDecoder
  Model C    60,166,145  SimpleCNNSpatial + GatedFusion + LSTM+MHCA
  Model D    54,940,161  ResNet101Spatial + GatedFusion + LSTM+MHCA
  Model E   182,094,337  ConvNeXt-Base + MUTANFusion + LSTM+MHCA
  Model F   183,151,105  BUTD(Faster R-CNN) + MUTANFusion + LSTM+MHCA


---
# 3. Data Preparation


## 3.1 Download Data

### COCO Images (required for all models)
```bash
# Create directories
mkdir -p data/raw/images

# Train 2014 (~13 GB)
wget http://images.cocodataset.org/zips/train2014.zip -P data/raw/
unzip data/raw/train2014.zip -d data/raw/

# Val 2014 (~6 GB)
wget http://images.cocodataset.org/zips/val2014.zip -P data/raw/
unzip data/raw/val2014.zip -d data/raw/
```

### VQA-E Annotations (required)
```bash
mkdir -p data/vqa_e
# Download from: https://github.com/liqing-ustc/VQA-E
# Place: data/vqa_e/VQA-E_train_set.json
#        data/vqa_e/VQA-E_val_set.json
```

### VQA v2.0 Annotations (for --mix_vqa pretraining, Tier D2)
```bash
mkdir -p data/raw/vqa_data_json
# Questions
wget https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Train_mscoco.zip
unzip v2_Questions_Train_mscoco.zip -d data/raw/vqa_data_json/
# Annotations
wget https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Train_mscoco.zip
unzip v2_Annotations_Train_mscoco.zip -d data/raw/vqa_data_json/
```


## 3.2 Build Vocabulary

Builds `vocab_questions.json` and `vocab_answers.json` from VQA-E training data.
Special tokens: `<pad>=0`, `<start>=1`, `<end>=2`, `<unk>=3`


In [7]:
import subprocess
result = subprocess.run(
    ['python', 'src/scripts/1_build_vocab.py'],
    capture_output=True, text=True, cwd='.'
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)



Reading VQA-E annotation file: data/vqa_e/VQA-E_train_set.json
Loaded 181298 annotations.

1. Building question vocabulary...
 -> Tokenizing...
 -> Filtering (threshold=3)...
 -> Done. Vocab size: 4546
   Vocab size: 4546 | Saved to: data/processed/vocab_questions.json

2. Building answer vocabulary (answer + explanation)...
 -> Tokenizing...
 -> Filtering (threshold=3)...
 -> Done. Vocab size: 8648
   Vocab size: 8648 | Saved to: data/processed/vocab_answers.json

Sample answer texts:
  Q: What is the green stuff?
  A: broccoli because Closeup of bins of food that include broccoli and bread.

  Q: What is in front of the giraffes?
  A: tree because Two giraffes standing in a tree filled area.

  Q: What do these giraffes have in common?
  A: eating because A giraffe eating food from the top of the tree.




In [17]:
# Verify vocab files and print statistics
import json

for name, path in [('Questions', 'data/processed/vocab_questions.json'),
                   ('Answers',   'data/processed/vocab_answers.json')]:
    with open(path) as f:
        v = json.load(f)
    print(f'{name} vocab: {len(v["word2idx"]):,} tokens')


Questions vocab: 4,546 tokens
Answers vocab: 8,650 tokens


## 3.3 (Optional) Filter Hallucinations — Tier D5

Removes VQA-E explanations with named entities, copy-paste patterns, or
degenerate repetitions. Reduces dataset by ~5-10% while improving quality.

**Requires**: `pip install spacy && python -m spacy download en_core_web_sm`


In [9]:
# Run hallucination filter (requires spaCy)
import subprocess
result = subprocess.run([
    'python', 'src/scripts/filter_hallucinations.py',
    '--input',  'data/vqa_e/VQA-E_train_set.json',
    '--output', 'data/vqa_e/VQA-E_train_filtered.json',
    '--report',
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('Error (spaCy not installed?):', result.stderr[:300])
    print('Using unfiltered data instead.')


Loading spaCy en_core_web_sm ...
Loading data/vqa_e/VQA-E_train_set.json ...
Original  : 181,298
Kept      : 172,810  (95.3%)
Removed   : 8,488  (4.7%)

Removal reasons:
  named_entity              : 8,416
  degenerate_repetition     : 36
  copy_of_question          : 23
  too_short                 : 13

Saved filtered annotations → data/vqa_e/VQA-E_train_filtered.json



## 3.4 (Optional) Generate Synthetic QA — Tier D5

Generates ~50K template-based QA pairs from COCO object annotations.
No LLM required — uses hand-crafted templates (existence, count, location).


In [16]:
# Generate synthetic QA from COCO instances
import subprocess, os
instances_json = 'data/raw/vqa_data_json/annotations/instances_train2014.json'
if os.path.exists(instances_json):
    result = subprocess.run([
        'python', 'src/scripts/generate_synthetic_qa.py',
        '--instances_json', instances_json,
        '--output', 'data/vqa_e/VQA-E_synthetic.json',
        '--max_total', '50000',
    ], capture_output=True, text=True)
    print(result.stdout)
else:
    print(f'⚠️  {instances_json} not found — skipping synthetic generation')


Loading data/raw/vqa_data_json/annotations/instances_train2014.json ...
Generated 50,000 synthetic QA pairs → data/vqa_e/VQA-E_synthetic.json
To use: concatenate with VQA-E_train_set.json before training.



## 3.5 Extract BUTD Features (Model F only)

Pre-extracts Faster R-CNN ResNet50 FPN v2 RoI features from all COCO images.
**Run once, ~3-4 hours on GPU. Skip if you only train Models A-E.**

Output: `data/butd_features/{image_id}.pt` — each file contains `{'feat': (k, 1029)}`
- 1024 dims: ResNet50 FPN box_head output (region semantic features)
- 5 dims: normalized spatial [x1/W, y1/H, x2/W, y2/H, area/(W·H)]
- k = 36 regions per image (top-36 RPN proposals by objectness score)


In [ ]:
# Extract BUTD features for Model F
# ⚠️  Takes ~3-4 hours for full COCO — use --max_images 100 for a quick test
import subprocess

EXTRACT_FULL = False   # ← set True when you want to run the full extraction

cmd = [
    'python', 'src/scripts/extract_butd_features.py',
    '--splits', 'train2014', 'val2014',
    '--image_dir', 'data/raw',
    '--output_dir', 'data/butd_features',
    '--top_k', '36',
]
if not EXTRACT_FULL:
    cmd += ['--max_images', '100']   # quick test
    print('Running quick test (100 images). Set EXTRACT_FULL=True for full run.')

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-2000:] if result.stdout else '')
if result.returncode != 0:
    print('Error:', result.stderr[:500])


---
# 4. Training

## Script-First Approach

> **For real training, use the shell scripts — not this notebook.**
>
> ```bash
> # Step 0: benchmark your GPU to find optimal batch size
> python src/scripts/benchmark_5070ti.py --quick
>
> # Model E (ConvNeXt) — full 4-phase training
> BATCH_SIZE=128 WANDB=1 bash train_model_e.sh
>
> # Model F (BUTD) — requires extract_butd_features.py first
> BATCH_SIZE=192 WANDB=1 bash train_model_f.sh
>
> # Single phase only:
> BATCH_SIZE=128 bash train_model_e.sh 1   # Phase 1 only
> ```

The cells below are **reference implementations** — useful for understanding flags,
debugging individual phases, or running quick experiments on a subset of data.

## Training Phases

| Phase | Epochs | LR | Key Purpose |
|-------|--------|----|-------------|
| 1 | 10 | 1e-3 | Learn all new modules; ConvNeXt frozen at ImageNet weights |
| 2 | 5 | 5e-4 | Unfreeze ConvNeXt top blocks (10× lower LR for CNN) |
| 3 | 5 | 2e-4 | Scheduled sampling: reduce exposure bias |
| 4 | 3 | 5e-5 | SCST RL: directly maximize BLEU-4 via REINFORCE |

## Hardware Profile (RTX 5070 Ti)

| Setting | Model E | Model F | Notes |
|---------|---------|---------|-------|
| `--batch_size` | 128 | 192 | Surrogate benchmark: 27%/20% VRAM — real ConvNeXt higher |
| `--num_workers` | 12 | 12 | 28 cores available; 12 optimal for I/O-bound loading |
| AMP | BFloat16 | BFloat16 | Auto-enabled (SM 12.0); no GradScaler needed |
| `torch.compile` | ON | ON | Default; speeds up MUTAN+MHCA ops |
| `--accum_steps` | 1 | 1 | Increase to 2 if OOM with ConvNeXt unlocked |


## 4.1 Quick Sanity Check (100 samples, 2 epochs)

Always run this first to verify the full pipeline compiles and runs without error.
Uses a small subset of data — not for measuring accuracy.

> Set `MODEL = 'E'` to test the production model (ConvNeXt + MHCA + MUTAN).  
> Use `--max_train_samples 100 --max_val_samples 50` for fast sanity check.


In [ ]:
import subprocess

MODEL        = 'E'     # Production model — test the full expansion stack
BATCH_SIZE   = 16      # Small batch for sanity check (fits any GPU)
MAX_SAMPLES  = 100

cmd = [
    'python', 'src/train.py',
    '--model',              MODEL,
    '--epochs',             '2',
    '--lr',                 '1e-3',
    '--batch_size',         str(BATCH_SIZE),
    '--max_train_samples',  str(MAX_SAMPLES),
    '--max_val_samples',    '50',
    '--use_mutan',
    '--layer_norm',
    '--q_highway',
    '--glove', '--glove_dim', '300',
    '--focal',
    '--curriculum',
    '--coverage',
    '--augment',
    '--num_workers',        '4',
    '--no_compile',   # disable compile for quick test (avoids Triton warmup)
    '--phase',        '1',
]
print('Running sanity check:', ' '.join(cmd))
result = subprocess.run(cmd, text=True)
print('\nExit code:', result.returncode)


## 4.2 Phase 1 — Baseline Training (10 epochs)

ConvNeXt frozen at ImageNet weights. Only the new modules learn:
MHCA attention, MUTAN fusion, LayerNorm-LSTM, BiLSTM question encoder,
SequenceFocalLoss, question-type curriculum.

**For real training use the shell script** — the cell below is a reference.

```bash
# Recommended for RTX 5070 Ti:
BATCH_SIZE=128 bash train_model_e.sh 1
```


In [ ]:
# Phase 1: mix_vqa ON, curriculum OFF (blocked), focal ON
MODEL, BATCH_SIZE, NUM_WORKERS = 'E', 128, 12

cmd_p1 = [
    'python', 'src/train.py',
    '--model', MODEL, '--epochs', '15', '--lr', '1e-3',
    '--warmup_epochs', '3', '--batch_size', str(BATCH_SIZE),
    '--num_workers', str(NUM_WORKERS),
    '--use_mutan', '--layer_norm', '--dropconnect',
    '--q_highway', '--char_cnn', '--glove', '--glove_dim', '300',
    '--coverage', '--augment',
    '--mix_vqa', '--mix_vqa_fraction', '0.7',   # Phase 1 only
    '--focal',                                   # active (no PGN)
    # '--curriculum' intentionally absent — blocked by --mix_vqa anyway
    '--dropout', '0.3', '--weight_decay', '1e-5',
    '--grad_clip', '5.0', '--label_smoothing', '0.1',
    '--early_stopping', '5', '--phase', '1',
]
print('Phase 1:'); print(' \\\n  '.join(cmd_p1))


In [ ]:
import subprocess
# Runs Phase 1 — expects cmd_p1 to be defined in the cell above
result = subprocess.run(cmd_p1, text=True)
print('Exit code:', result.returncode)


## 4.3 Phase 2 — CNN Fine-tuning (5 epochs)

**Only for Models B, D, E** — unfreezes ResNet/ConvNeXt top layers with
differential LR (backbone LR = base LR × 0.1) to avoid catastrophic forgetting.

**Models A, C**: skip Phase 2 (scratch CNN, already fully trainable).


In [ ]:
# Phase 2: mix_vqa OFF, curriculum ON, focal ON
MODEL, BATCH_SIZE = 'E', 128

cmd_p2 = [
    'python', 'src/train.py',
    '--model', MODEL, '--epochs', '12', '--lr', '5e-4',
    '--batch_size', str(BATCH_SIZE), '--num_workers', '12',
    '--resume', 'checkpoints/model_e_resume.pth',
    '--use_mutan', '--layer_norm', '--dropconnect',
    '--q_highway', '--char_cnn', '--glove', '--glove_dim', '300',
    '--coverage', '--augment',
    '--curriculum',   # NOW active — mix_vqa is gone
    '--focal',
    '--finetune_cnn', '--cnn_lr_factor', '0.1',
    '--dropout', '0.3', '--weight_decay', '1e-5',
    '--grad_clip', '5.0', '--label_smoothing', '0.1',
    '--early_stopping', '3', '--phase', '2',
]
print('Phase 2:'); print(' \\\n  '.join(cmd_p2))
# Uncomment to run: import subprocess; subprocess.run(cmd_p2, text=True)


## 4.4 Phase 3 — Scheduled Sampling (5 epochs)

Reduces **exposure bias**: during training the model always sees ground-truth
tokens (teacher forcing), but at inference it must predict from its own output.

Scheduled sampling gradually replaces GT tokens with the model's own predictions:
```
epsilon(epoch) = k / (k + exp(epoch / k))   (inverse-sigmoid decay)
epsilon ≈ 1.0 at epoch 0  →  mostly teacher forcing
epsilon ≈ 0.5 at epoch k  →  50% teacher forcing
```


In [ ]:
# Phase 3: mix_vqa OFF, curriculum ON, focal ON
MODEL = 'E'

cmd_p3 = [
    'python', 'src/train.py',
    '--model', MODEL, '--epochs', '7', '--lr', '2e-4',
    '--batch_size', '128', '--num_workers', '12',
    '--resume', 'checkpoints/model_e_resume.pth',
    '--use_mutan', '--layer_norm', '--dropconnect',
    '--q_highway', '--char_cnn', '--glove', '--glove_dim', '300',
    '--coverage', '--augment',
    '--curriculum', '--focal',
    '--finetune_cnn', '--cnn_lr_factor', '0.1',
    '--scheduled_sampling', '--ss_k', '5',
    '--dropout', '0.3', '--weight_decay', '1e-5',
    '--grad_clip', '5.0', '--label_smoothing', '0.1',
    '--early_stopping', '3', '--phase', '3',
]
print('Phase 3:'); print(' \\\n  '.join(cmd_p3))
# Uncomment to run: import subprocess; subprocess.run(cmd_p3, text=True)


## 4.5 Phase 4 — SCST Reinforcement Learning (3 epochs)

Self-Critical Sequence Training (Rennie et al. 2017): REINFORCE with the
model's own greedy output as the baseline.

```
r_sample  = BLEU-4(sampled_output, reference)     # stochastic decode
r_greedy  = BLEU-4(greedy_output,  reference)     # deterministic baseline
loss_RL   = -mean((r_sample - r_greedy) · Σ log p_t)
loss_total = CE_loss + λ_scst · loss_RL            (λ_scst = 0.5)
```

**Only run after Phase 3.** SCST on an untrained model is unstable.


In [ ]:
# Phase 4: mix_vqa OFF, curriculum ON, focal OFF (plain CE for SCST stability)
MODEL = 'E'

cmd_p4 = [
    'python', 'src/train.py',
    '--model', MODEL, '--epochs', '3', '--lr', '5e-5',
    '--batch_size', '128', '--num_workers', '12',
    '--resume', 'checkpoints/model_e_resume.pth',
    '--use_mutan', '--layer_norm', '--dropconnect',
    '--q_highway', '--char_cnn', '--glove', '--glove_dim', '300',
    '--coverage', '--augment',
    '--curriculum',
    # NO --focal: SCST uses plain CE as supervised anchor.
    # Focal re-weighting destabilizes REINFORCE gradient variance.
    '--finetune_cnn', '--cnn_lr_factor', '0.1',
    '--scst', '--scst_lambda', '0.5',
    '--dropout', '0.3', '--weight_decay', '1e-5',
    '--grad_clip', '5.0', '--phase', '4',
]
print('Phase 4:'); print(' \\\n  '.join(cmd_p4))
# Uncomment to run: import subprocess; subprocess.run(cmd_p4, text=True)


## 4.3 Train Model F (BUTD Pre-extracted Features)

Model F achieves the highest accuracy ceiling — uses Faster R-CNN object features
instead of a CNN backbone. Pre-extraction is a one-time cost (~3-4h for full COCO).

**Prerequisites:**
```bash
# 1. Extract BUTD features (run once, then cached as .pt files)
python src/scripts/extract_butd_features.py \
    --image_dir data/raw \
    --output_dir data/butd_features \
    --splits train2014 val2014

# 2. Full training (use the dedicated script):
BATCH_SIZE=192 WANDB=1 bash train_model_f.sh
```

Model F differences from E:
- No Phase 2 CNN fine-tuning (features are pre-extracted and frozen)
- `butd_collate_fn` returns 4-tuple `(feats, q, a, img_mask)` — variable-k masking
- Higher batch size safe (no CNN in forward pass during training)
- `img_mask` prevents MHCA from attending to padded zero-feature rows


In [ ]:
# Model F — Phase 1 command (reference)
# For full training: BATCH_SIZE=192 bash train_model_f.sh

cmd_f = [
    'python', 'src/train.py',
    '--model',          'F',
    '--epochs',         '10',
    '--lr',             '1e-3',
    '--batch_size',     '192',     # no CNN → higher batch fits easily
    '--num_workers',    '12',
    '--warmup_epochs',  '3',
    '--use_mutan', '--layer_norm', '--dropconnect',
    '--q_highway', '--char_cnn',
    '--glove', '--glove_dim', '300',
    '--focal', '--curriculum', '--coverage',
    '--mix_vqa', '--mix_vqa_fraction', '0.7',
    '--augment',
    '--dropout', '0.3', '--weight_decay', '1e-5',
    '--grad_clip', '5.0', '--label_smoothing', '0.1',
    '--early_stopping', '5',
    '--phase', '1',
]
print('Model F Phase 1 command:')
print(' \\\n  '.join(cmd_f))

# Uncomment to run Model F Phase 1 from notebook:
# import subprocess; subprocess.run(cmd_f, text=True)


## 4.4 Shell Scripts — Primary Training Interface

```
train_model_e.sh     ConvNeXt (Model E) — 4-phase training
train_model_f.sh     BUTD (Model F) — 4-phase training
```

```bash
# ── Quick start ──────────────────────────────────────────────────────────────
# Step 1: Benchmark your hardware
python src/scripts/benchmark_5070ti.py --quick

# Step 2: Train Model E (all 4 phases, ~16-20 hours on RTX 5070 Ti)
BATCH_SIZE=128 WANDB=1 bash train_model_e.sh

# Step 3 (optional): Train Model F (requires BUTD features)
BATCH_SIZE=192 WANDB=1 bash train_model_f.sh

# ── Single phase ─────────────────────────────────────────────────────────────
BATCH_SIZE=128 bash train_model_e.sh 1    # Phase 1 only
BATCH_SIZE=128 bash train_model_e.sh 2    # Phase 2 (resume required)

# ── Resume from checkpoint ────────────────────────────────────────────────────
# Checkpoints: model_e_resume.pth (latest), model_e_best.pth (best val loss)
# Just run the next phase — it auto-loads from model_e_resume.pth

# ── Monitoring ────────────────────────────────────────────────────────────────
# W&B: add WANDB=1 prefix to any command above
# Local logs: checkpoints/history_model_e.json
```

**Expected training time (RTX 5070 Ti, batch 128):**

| Phase | Time |
|-------|------|
| Phase 1 (10ep, frozen CNN) | ~3-4h |
| Phase 2 (5ep, fine-tune CNN) | ~2-3h |
| Phase 3 (5ep, scheduled sampling) | ~2-3h |
| Phase 4 (3ep, SCST RL) | ~1-2h |
| **Total Model E** | **~8-12h** |
| **Total Model F** (no CNN fine-tune) | **~5-8h** |


---
# 5. Evaluation

## Metrics

| Metric | What it measures |
|--------|-----------------|
| **VQA Accuracy** | Soft matching against 10 human annotations (official VQA metric) |
| **Exact Match** | Exact string match after normalization |
| **BLEU-1/2/3/4** | N-gram precision (higher n → stricter fluency) |
| **METEOR** | Unigram F-score with stemming and synonymy (via WordNet) |
| **ROUGE-L** | Longest common subsequence recall (explanation coherence) |


In [ ]:
# Run evaluation on a saved checkpoint
import subprocess, json

MODEL      = 'E'
CKPT_EPOCH = 10    # which epoch checkpoint to evaluate
BEAM_WIDTH = 3     # 1=greedy, 3=beam search (better quality, slower)

ckpt_path = f'checkpoints/model_{MODEL.lower()}_epoch{CKPT_EPOCH}.pth'

result = subprocess.run([
    'python', 'src/evaluate.py',
    '--model_type', MODEL,
    '--checkpoint', ckpt_path,
    '--beam_width', str(BEAM_WIDTH),
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print('Error:', result.stderr[-1000:])


In [ ]:
# Evaluate all available models and collect results
import subprocess, os, re

results = {}
for model in ['A', 'B', 'C', 'D', 'E']:
    for epoch in [10, 15, 20]:
        ckpt = f'checkpoints/model_{model.lower()}_epoch{epoch}.pth'
        if not os.path.exists(ckpt):
            continue
        r = subprocess.run([
            'python', 'src/evaluate.py',
            '--model_type', model, '--checkpoint', ckpt,
            '--beam_width', '3',
        ], capture_output=True, text=True)
        # Parse BLEU-4 from output
        m = re.search(r'BLEU-4[^:]*:\s*([0-9.]+)', r.stdout)
        if m:
            results[f'{model}_ep{epoch}'] = float(m.group(1))
            print(f'Model {model} epoch {epoch}: BLEU-4 = {m.group(1)}')

if results:
    best = max(results, key=results.get)
    print(f'\nBest: {best} → BLEU-4 = {results[best]:.4f}')


In [ ]:
# Inline evaluation (no subprocess) — load model directly
import sys; sys.path.insert(0, 'src')
import torch
from evaluate import evaluate

MODEL = 'E'
CKPT  = f'checkpoints/model_{MODEL.lower()}_best.pth'

if os.path.exists(CKPT):
    metrics = evaluate(
        model_type  = MODEL,
        checkpoint  = CKPT,
        num_samples = 500,    # set None for full validation set
        beam_width  = 3,
    )
else:
    print(f'Checkpoint not found: {CKPT}')
    print('Train the model first (Section 4).')


---
# 6. Visualization


## 6.1 Training Curves


In [ ]:
import json, os
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'A':'#e74c3c','B':'#e67e22','C':'#2ecc71','D':'#3498db','E':'#9b59b6','F':'#1abc9c'}

for model in ['A','B','C','D','E','F']:
    path = f'checkpoints/history_model_{model.lower()}.json'
    if not os.path.exists(path):
        continue
    with open(path) as f:
        h = json.load(f)
    epochs = range(1, len(h['train_loss'])+1)
    c = colors.get(model, '#888')
    axes[0].plot(epochs, h['train_loss'], label=f'Model {model}', color=c)
    axes[1].plot(epochs, h['val_loss'],   label=f'Model {model}', color=c, linestyle='--')

for ax, title in zip(axes, ['Train Loss', 'Val Loss']):
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Training Curves — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('checkpoints/training_curves.png', dpi=150)
plt.show()
print('Saved → checkpoints/training_curves.png')


## 6.2 Attention Heatmaps (Models C/D/E/F)

Visualizes where the model looks in the image (img_alpha) and question
(q_alpha from MHCA) while generating each answer token.


In [ ]:
import subprocess
result = subprocess.run([
    'python', 'src/visualize.py',
    '--model_type', 'E',
    '--epoch', '20',
    '--sample_idx', '0',
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('Error:', result.stderr[:500])


In [ ]:
# Inline attention visualization
import sys; sys.path.insert(0, 'src')
import torch, json, os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image
from vocab import Vocabulary
from inference import load_model_from_checkpoint, get_model
from dataset import VQAEDataset

def visualize_attention(model_type, checkpoint, sample_idx=0, beam_width=1):
    vocab_q = Vocabulary(); vocab_q.load('data/processed/vocab_questions.json')
    vocab_a = Vocabulary(); vocab_a.load('data/processed/vocab_answers.json')
    dataset = VQAEDataset(
        image_dir='data/raw/val2014',
        vqa_e_json_path='data/vqa_e/VQA-E_val_set.json',
        vocab_q=vocab_q, vocab_a=vocab_a, split='val2014'
    )
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    img_tensor, q_tensor, a_tensor = dataset[sample_idx]
    ann = dataset.annotations[sample_idx]
    print(f"Question : {ann['question']}")
    print(f"Reference: {ann.get('multiple_choice_answer','')} because {ann.get('explanation',[''])[0][:80]}")
    print('Checkpoint required — run training first to generate a checkpoint.')

# Call with your checkpoint:
# visualize_attention('E', 'checkpoints/model_e_best.pth', sample_idx=5)
print('Attention visualization ready — provide a checkpoint path to activate.')


## 6.3 Sample Predictions


In [ ]:
import sys; sys.path.insert(0, 'src')
import torch, os
from vocab import Vocabulary
from dataset import VQAEDataset

def show_predictions(model_type, checkpoint_path, n_samples=5, beam_width=3):
    if not os.path.exists(checkpoint_path):
        print(f'Checkpoint not found: {checkpoint_path}')
        return
    from inference import load_model_from_checkpoint, batch_beam_search_decode_with_attention
    from inference import batch_greedy_decode

    device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    vocab_q  = Vocabulary(); vocab_q.load('data/processed/vocab_questions.json')
    vocab_a  = Vocabulary(); vocab_a.load('data/processed/vocab_answers.json')
    dataset  = VQAEDataset(
        'data/raw/val2014', 'data/vqa_e/VQA-E_val_set.json',
        vocab_q, vocab_a, split='val2014'
    )
    model = load_model_from_checkpoint(model_type, checkpoint_path,
                                       len(vocab_q), len(vocab_a)).to(device)
    model.eval()
    sep = '=' * 70
    print(f'\n{sep}')
    print(f'  Model {model_type} — {os.path.basename(checkpoint_path)}')
    print(sep)
    for i in range(n_samples):
        img, q_t, a_t = dataset[i]
        ann = dataset.annotations[i]
        imgs = img.unsqueeze(0).to(device)
        qs   = q_t.unsqueeze(0).to(device)
        pred = batch_greedy_decode(model, model_type, imgs, qs, vocab_a, max_len=40)
        ref  = ann.get('multiple_choice_answer','') + ' because ' + (ann.get('explanation',[''])[0])[:80]
        print(f'  Q : {ann["question"]}')
        print(f'  GT: {ref.strip()}')
        print(f'  PR: {pred[0].strip()}')
        print('  ' + '-' * 60)

# show_predictions('E', 'checkpoints/model_e_best.pth')
print('show_predictions() ready — provide a checkpoint to call it.')


## 6.4 Model Comparison Table


In [ ]:
import subprocess, os, re

models_to_compare = [m for m in ['A','B','C','D','E','F']
    if os.path.exists(f'checkpoints/model_{m.lower()}_best.pth')]

if models_to_compare:
    result = subprocess.run([
        'python', 'src/compare.py',
        '--models', ','.join(models_to_compare),
        '--epoch', '20',
    ], capture_output=True, text=True)
    print(result.stdout)
else:
    print('No checkpoints found yet. Train models first.')
    print('Expected: checkpoints/model_{a,b,c,d,e,f}_best.pth')


---
# 7. Advanced Techniques


## 7.1 Sequence Focal Loss (Tier D3)

Drop-in replacement for CrossEntropyLoss. Enable with `--focal --focal_gamma 2.0`.


In [ ]:
import sys; sys.path.insert(0, 'src')
import torch
from training.losses import SequenceFocalLoss
import torch.nn as nn

# Compare Focal vs CE on a sequence with rare and common tokens
torch.manual_seed(42)
B, T, V = 4, 20, 3000
logits  = torch.randn(B*T, V)
targets = torch.randint(0, V, (B*T,))
targets[::5] = 0   # 20% padding

ce_loss    = nn.CrossEntropyLoss(ignore_index=0)(logits, targets)
focal_loss = SequenceFocalLoss(gamma=2.0)(logits, targets)

print(f'CrossEntropyLoss  : {ce_loss.item():.4f}')
print(f'SequenceFocalLoss : {focal_loss.item():.4f}  (γ=2.0)')
print()
print('Focal loss is lower for well-predicted tokens (p_t high)')
print('and comparable for hard tokens (p_t low). Both converge to')
print('the same solution but Focal focuses gradient on hard tokens.')


## 7.2 Question-Type Curriculum (Tier D4)

Enable with `--curriculum`. Trains on easy question types first:
binary (0-25%) → color/count (25-50%) → what/where (50-75%) → why/how (full).


In [ ]:
import sys; sys.path.insert(0, 'src')
from training.curriculum import classify_question_type, CurriculumSampler
import matplotlib.pyplot as plt

# Show the question-type classification
examples = [
    'Is the dog running?',
    'Are there any trees?',
    'What color is the car?',
    'How many people are there?',
    'What is on the table?',
    'Where is the cat sitting?',
    'Which direction is the bus going?',
    'Why is the man smiling?',
    'How was the food prepared?',
]
tier_names = {0:'Binary', 1:'Color', 2:'Count', 3:'What', 4:'Where/Which', 5:'Why/How'}
print(f"{'Question':<45} Tier  Type")
print('-'*65)
for q in examples:
    t = classify_question_type(q)
    print(f'{q:<45}  {t}   {tier_names[t]}')


In [ ]:
# Simulate curriculum progress over 10 epochs
mock_anns = [{'question': q} for q in examples * 100]

from training.curriculum import compute_question_type_scores
scores  = compute_question_type_scores(mock_anns)
sampler = CurriculumSampler(scores, epoch=0, total_epochs=10)

print('Epoch  Stage   Pool size  Pct')
for ep in range(10):
    sampler.set_epoch(ep)
    pct = len(sampler) / len(scores) * 100
    stage = 'Binary' if ep < 2 else ('Color/Count' if ep < 5 else ('What/Where' if ep < 7 else 'Full'))
    print(f'  {ep+1:2d}   {stage:<12}  {len(sampler):>5}     {pct:.0f}%')


## 7.3 Mixed-Ratio Pretraining (Tier D2)

Enable with `--mix_vqa --mix_vqa_fraction 0.7` in Phase 1 only.

**Why**: VQA v2.0 answers are 1-3 tokens. Training on VQA v2.0 alone induces
length bias — the LSTM learns to emit `<end>` after ~2 tokens, contradicting
VQA-E's 5-20 token explanation objective.

**How**: WeightedRandomSampler gives VQA-E 3× oversampling relative to its
natural frequency in the combined dataset.


In [ ]:
import sys; sys.path.insert(0, 'src')
import torch
from torch.utils.data import Dataset, WeightedRandomSampler, ConcatDataset
from dataset import build_mixed_sampler

# Simulate the mixing behavior
class FakeDataset(Dataset):
    def __init__(self, n, label): self.n = n; self.label = label
    def __len__(self): return self.n
    def __getitem__(self, i): return i

vqa_v2  = FakeDataset(660_000, 'VQA v2.0')   # ~660K samples
vqa_e   = FakeDataset(210_000, 'VQA-E')       # ~210K samples

concat, sampler = build_mixed_sampler(vqa_v2, vqa_e, vqa_fraction=0.7)
print(f'ConcatDataset size  : {len(concat):,}')
print(f'Sampler num_samples : {sampler.num_samples:,}')
print(f'Weight VQA v2.0/sample : {0.7/660_000:.2e}  ({0.7:.0%} of batches)')
print(f'Weight VQA-E/sample    : {0.3/210_000:.2e}  ({0.3:.0%} of batches)')
print(f'VQA-E oversampling     : {(0.3/210_000) / (0.7/660_000):.1f}×')


## 7.4 ConceptNet GNN (Tier 9 — Optional)

Enriches word embeddings with commonsense knowledge from ConceptNet via GCN.
Falls back to a 2-layer MLP if `torch_geometric` is not installed.


In [ ]:
import sys; sys.path.insert(0, 'src')
import torch
from models.concept_gnn import ConceptGNN

# Demo: enrich word embeddings
gnn = ConceptGNN(vocab_size=8000, embed_dim=512, gcn_dim=256)

word_ids = torch.randint(4, 8000, (2, 15))   # (batch=2, seq_len=15)
enriched = gnn(word_ids)                      # (2, 15, 256)
print(f'Input  word_ids : {word_ids.shape}')
print(f'Output enriched : {enriched.shape}')
print()
print('Integration: pass to QuestionEncoder as enriched_emb argument')
print('after building the co-occurrence graph from training annotations.')


## 7.5 CSS Counterfactual Augmentation (Tier 6)

Enable with `--css --css_lambda 0.5`. Creates two counterfactual versions
of each batch to build visual and linguistic grounding:

- **Visual CF**: zero the top-k attended image regions → model must explain
  without the most attended object → forces multi-region reasoning
- **Linguistic CF**: replace content words in question with `<unk>` →
  model must not rely on question shortcuts

Loss = CE + λ_css · max(0, margin - ||f_real - f_cf||₂)


## 7.6 SCST Reinforcement Learning (Tier 8)

Enable with `--scst --scst_lambda 0.5` in Phase 4.

**Key insight**: CE loss rewards log-probability of the GT token at every
position, but BLEU-4 (the actual metric) depends on n-gram precision across
the full sequence. SCST directly optimizes the sequence-level metric:

```
r_sample = BLEU-4(model_sample, reference)   ← roll out stochastic decode
r_greedy = BLEU-4(greedy_output, reference)  ← baseline (no gradient)
REINFORCE loss = -(r_sample - r_greedy) · Σ_t log p(y_t^sample)
```

The baseline `r_greedy` reduces variance without biasing the gradient.


---
# 8. Full Model Benchmark


In [ ]:
# Load training history for all available models
import json, os
import matplotlib.pyplot as plt
import numpy as np

# Expected results after full training (fill in after actual runs):
expected = {
    # Baselines (pre-expansion)
    'A': {'bleu4': 5.1,  'meteor': 22.3, 'rouge_l': 31.2, 'label': 'A: SimpleCNN'},
    'B': {'bleu4': 7.4,  'meteor': 27.8, 'rouge_l': 36.9, 'label': 'B: ResNet101'},
    'C': {'bleu4': 8.9,  'meteor': 30.4, 'rouge_l': 39.1, 'label': 'C: SimpleCNN+Attn'},
    'D': {'bleu4': 11.6, 'meteor': 35.9, 'rouge_l': 42.7, 'label': 'D: ResNet+Attn (baseline)'},
    # Expansion targets
    'E': {'bleu4': None, 'meteor': None, 'rouge_l': None, 'label': 'E: ConvNeXt+MHCA+MUTAN'},
    'F': {'bleu4': None, 'meteor': None, 'rouge_l': None, 'label': 'F: BUTD+MHCA+MUTAN'},
}
# Fill E/F after training:
#   expected['E'] = {'bleu4': 18.5, 'meteor': 44.2, 'rouge_l': 51.3, 'label': 'E: ConvNeXt+MHCA+MUTAN'}
#   expected['F'] = {'bleu4': 22.1, 'meteor': 48.7, 'rouge_l': 55.8, 'label': 'F: BUTD (best)'}

history_dir = 'checkpoints'
histories = {}
for m in ['A', 'B', 'C', 'D', 'E', 'F']:
    path = os.path.join(history_dir, f'history_model_{m.lower()}.json')
    if os.path.exists(path):
        with open(path) as f:
            histories[m] = json.load(f)
        print(f"Loaded: {path}")
    else:
        print(f"Not found: {path} (run training first)")

if histories:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = {'A': '#aaa', 'B': '#888', 'C': '#666', 'D': '#444', 'E': '#e05c00', 'F': '#0072b2'}

    for m, hist in histories.items():
        ep = range(1, len(hist.get('val_loss', [])) + 1)
        lw = 2.5 if m in ('E', 'F') else 1.0
        ls = '-' if m in ('E', 'F') else '--'
        label = expected.get(m, {}).get('label', m)
        if 'val_loss' in hist:
            axes[0].plot(ep, hist['val_loss'], color=colors.get(m, 'gray'),
                         linewidth=lw, linestyle=ls, label=label)
        if 'bleu4' in hist:
            axes[1].plot(ep, hist['bleu4'], color=colors.get(m, 'gray'),
                         linewidth=lw, linestyle=ls, label=label)

    axes[0].set(title='Validation Loss', xlabel='Epoch', ylabel='Loss')
    axes[1].set(title='BLEU-4', xlabel='Epoch', ylabel='BLEU-4 (%)')
    for ax in axes:
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('checkpoints/training_curves.png', dpi=150)
    plt.show()
    print("Saved: checkpoints/training_curves.png")


In [ ]:
# Architecture ablation reference table (fill in after evaluation)

ablation = {
    'Model A (SimpleCNN, no attn)':        {'BLEU-4': '—', 'METEOR': '—', 'Notes': 'Baseline'},
    'Model B (ResNet101, no attn)':        {'BLEU-4': '—', 'METEOR': '—', 'Notes': 'Pretrained encoder'},
    'Model C (SimpleCNN + MHCA)':          {'BLEU-4': '—', 'METEOR': '—', 'Notes': 'Spatial attention'},
    'Model D (ResNet101 + MHCA)':          {'BLEU-4': '—', 'METEOR': '—', 'Notes': 'Pretrained + attn'},
    'Model E (ConvNeXt + MUTAN + MHCA)':   {'BLEU-4': '—', 'METEOR': '—', 'Notes': 'Full best model'},
    'Model F (BUTD + MUTAN + MHCA)':       {'BLEU-4': '—', 'METEOR': '—', 'Notes': 'Object-centric'},
}

print(f"{'Architecture':<42} {'BLEU-4':>8}  {'METEOR':>8}  Notes")
print('-'*80)
for arch, m in ablation.items():
    print(f'{arch:<42} {m["BLEU-4"]:>8}  {m["METEOR"]:>8}  {m["Notes"]}')

print()
print('Fill in BLEU-4 and METEOR after running evaluate.py on each checkpoint.')


---
# 9. Inference API

Run the trained model on arbitrary images and questions.


In [ ]:
import sys; sys.path.insert(0, 'src')
import torch
from PIL import Image
from torchvision import transforms
from vocab import Vocabulary

def load_inference_model(model_type, checkpoint_path):
    """Load a trained model ready for inference."""
    from inference import load_model_from_checkpoint
    device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    vocab_q = Vocabulary(); vocab_q.load('data/processed/vocab_questions.json')
    vocab_a = Vocabulary(); vocab_a.load('data/processed/vocab_answers.json')
    model   = load_model_from_checkpoint(
        model_type, checkpoint_path, len(vocab_q), len(vocab_a))
    model   = model.to(device).eval()
    return model, vocab_q, vocab_a, device

def answer_question(model, model_type, image_path, question_text,
                    vocab_q, vocab_a, device, beam_width=3):
    """Answer a question about an image."""
    from inference import batch_beam_search_decode_with_attention
    from inference import batch_greedy_decode
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    img = transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(device)
    q_ids = torch.tensor(vocab_q.numericalize(question_text)).unsqueeze(0).to(device)

    if beam_width <= 1:
        answer = batch_greedy_decode(model, model_type, img, q_ids, vocab_a)
    else:
        answer = batch_beam_search_decode_with_attention(
            model, model_type, img, q_ids, vocab_a, beam_width=beam_width)
    return answer[0] if isinstance(answer, list) else answer

print('Inference API ready.')
print('Usage:')
print('  model, vocab_q, vocab_a, device = load_inference_model("E", "checkpoints/model_e_best.pth")')
print('  answer = answer_question(model, "E", "path/to/image.jpg", "What is in the image?", ...)')


---
# 10. Quick Reference — All train.py Flags

## Full Flag Reference

```bash
python src/train.py \
  --model       E         # A/B/C/D (legacy) or E (ConvNeXt) or F (BUTD)
  --epochs      10
  --lr          1e-3
  --batch_size  128        # RTX 5070 Ti: 128 (E), 192 (F)
  --num_workers 12         # optimal for 28-core CPU

  # ── Expansion architecture flags ──────────────────────────────────────────
  --use_mutan              # MUTAN Tucker bilinear fusion (Tier 4)
  --layer_norm             # LayerNorm LSTM gates (Tier 1A)
  --dropconnect            # DropConnect h-to-h weights (Tier 1B)
  --q_highway              # Highway BiLSTM question encoder (Tier 7B)
  --char_cnn               # CharCNN character embeddings (Tier 7C)
  --glove --glove_dim 300  # GloVe 840B pretrained (Tier 0)
  --coverage               # Coverage mechanism anti-repeat (Tier 2)
  --pgn                    # Pointer-Generator Network (Tier 5)

  # ── Data / loss flags ─────────────────────────────────────────────────────
  --augment                # RandAugment + RandomErasing (Tier D1)
  --mix_vqa                # 70% VQAv2 + 30% VQA-E mixed (Tier D2)
  --focal                  # SequenceFocalLoss (Tier D3)
  --curriculum             # Question-type curriculum (Tier D4)
  --css --css_lambda 0.5   # CSS counterfactual augmentation (Tier 6)

  # ── Training regime ───────────────────────────────────────────────────────
  --warmup_epochs 3        # LR warmup
  --dropout       0.3
  --weight_decay  1e-5
  --grad_clip     5.0
  --label_smoothing 0.1
  --early_stopping  5
  --accum_steps     1      # gradient accumulation (increase if OOM)

  # ── Phase-specific ────────────────────────────────────────────────────────
  --finetune_cnn           # Phase 2: unfreeze CNN backbone
  --cnn_lr_factor 0.1      # Phase 2: CNN LR = base_lr × 0.1
  --scheduled_sampling     # Phase 3: mix teacher forcing / model output
  --ss_k 5                 # Phase 3: decay speed
  --scst                   # Phase 4: SCST RL
  --scst_lambda 0.5        # Phase 4: CE/RL mix ratio

  # ── Resume & checkpoint ───────────────────────────────────────────────────
  --resume  checkpoints/model_e_resume.pth
  --phase   1              # tag for W&B grouping

  # ── Tracking ──────────────────────────────────────────────────────────────
  --wandb                  # enable W&B
  --wandb_project vqa-e
  --wandb_run_name model_e_phase1
  --wandb_tags    "modelE,phase1,convnext"
  --no_compile             # disable torch.compile (use for quick tests)
```

## RTX 5070 Ti Cheat Sheet

```bash
# Benchmark (run once to find optimal batch size)
python src/scripts/benchmark_5070ti.py --quick

# Model E — full training
BATCH_SIZE=128 WANDB=1 bash train_model_e.sh

# Model F — full training (needs BUTD features first)
python src/scripts/extract_butd_features.py --splits train2014 val2014
BATCH_SIZE=192 WANDB=1 bash train_model_f.sh

# Evaluate best checkpoint
python src/evaluate.py --model_type E --checkpoint checkpoints/model_e_best.pth --beam_width 3

# Visualize attention
python src/visualize.py --model_type E --checkpoint checkpoints/model_e_best.pth --sample_idx 0
```


---

<p align='center'>
  <b>VQA-E Master Guide</b> — All-in-One Notebook<br>
  Models A–F · MHCA · MUTAN · PGN · CSS · SCST · Focal Loss · Curriculum
</p>
